# Retrieval architectures in the notebook prototype

This notebook records the interactive prototype. It defines the Explorer and vector search tools, Jev traversal, and guideline filtering. The measured experiment used five architectures and one model-only control. Its exact construction is in `retrieval_agents()` in [run_retrieval.py](run_retrieval.py).

The prototype requires a private topic tree and vector database during setup. Its demo cells launch agents and make provider calls. The historical prototype uses a moving model alias and its demonstration setup assigns the same fixed traversal to the GROVE and GROVE-Free variables. Use the runner for the measured GROVE-Free configuration. See the [artifact README](../README.md#use-the-notebooks) for setup and data.


In [ ]:
import asyncio
import base64
import json
import mimetypes
import os
import re
from pathlib import Path

import aisuite
import duckdb
import numpy as np
from dotenv import load_dotenv
from IPython.display import Markdown, display
from typesafe_sdk import AsyncTypeSafeClient, Noul, NoulCriteria, TypeSafeClient

from fast_track_lib import Clause, Policy, create_agent
from fast_track_lib.core.embeddings.models import AisuiteEmbeddingModel
from fast_track_lib.tools.basic import read_file, run_bash, write_file
from fast_track_lib.tracing import Tracer
from fast_track_lib.webdemo import AgentDemo

load_dotenv()
client = aisuite.Client()  # read_images and jev_explore use it

LLM_MODEL = "anthropic:claude-sonnet-4-6"
# shared by every agent below
AGENT_KWARGS = dict(model=LLM_MODEL, memory="dynamic",
                    memory_params={"window_size": 100000, "token_limit": 100000})

In [ ]:
# Body of Knowledge layout (see EXPLORER_PROMPT):
#   swebok-v4/root/      -> topic tree (INDEX.md + README.md per folder, content .md at the leaves)
#   swebok-v4/tables/    -> tables referenced as [Table N](tables/...)
#   swebok-v4/images/    -> images referenced as ![image](images/...)
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "swebok-v4").exists())
BOK_PATH = REPO_ROOT / "swebok-v4"
BOK_ROOT = BOK_PATH / "root"
READ_LIMIT = 4  # read_many batches with content files allowed before the explorer must write its answer


def check_valid_file(file_path: str, start_line: int = None, end_line: int = None):
    if not os.path.isfile(file_path):
        if os.path.isdir(file_path):
            available = os.listdir(file_path)
            raise PermissionError(
                f"You need to read a file, not a path! "
                f"Available files/folders in '{file_path}': {available}"
            )
        raise PermissionError("You need to read a file, not a path!")

def check_multiple_files(paths):
    any_error = []
    error_msg = []
    for path in paths:
        error = False
        try:
            check_valid_file(path)
        except Exception as e:
            error = True
            error_msg.append('Found an error for the file: {path} - {error}'.format(path = path,error = error))  
        any_error.append(error)
    if any(any_error):
        raise PermissionError('\n'.join(error_msg))
            
            
       
def read_many(paths: list[str]) -> str:
        """Read multiple files at a time and returns the full text

        Args:
            paths (List[str]): list of paths to be read

        Returns:
            content_text (str): full content text with references
        """
        content = []
        for path in paths:
            text = read_file(path)
            content.append(f'Here is the content for the file: {path}:\n{text}\n')
        content_text = '\n'.join(content)
        return content_text

def plan_tasks(plan: str):
    """
    Use this tool to plan you approach
    plan: str. Your plan to solve the problem
    """
    return plan

class ReadBudget:
    """Read is metered; a successful write refills the budget."""
    def __init__(self, limit=4):
        self.limit = limit
        self.count = 0

    def check_read(self, paths):#file_path: str, start_line=None, end_line=None):  # input on read_file
        if not all([re.search(r'INDEX\.md|README\.md|STYLES\.md', file_path) is not None for file_path in paths]):
            self.count += 1
        if self.count > self.limit:
            raise PermissionError(
                f"read_file is locked after {self.limit} reads. "
                "Now you must finish your task at once!"
            )

    def unlock(self, content):    # output on write_file — the hook
        self.count = 0


class ReadOnce:
    def __init__(self):
        self.visited = set()

    def __call__(self, paths):
        visited_text = "\n".join(self.visited)
        if set(paths).issubset(self.visited):
            file_path = "\n".join(paths)
            raise PermissionError(f"You already read the files:\n{file_path}\n"
                                  f"Either finish the task or read other files outside this list:\n{visited_text}")
        self.visited.update(paths)


class ReadFirst:
    def __init__(self, files):
        self.files = set(files)
        self.visited = set()

    def check_first(self, paths):
        self.visited.update(paths)
        condition = self.files.issubset(self.visited)
        if len(self.visited) <= len(self.files):
            condition = self.visited.issubset(self.files)
        if not condition:
            raise PermissionError(f"You need to read the following files first: {', '.join(self.files)}!")



def get_policy(final_tool="write_file"):
    """Explorer policy: plan → read → final_tool, in that order (final_tool is a dead end)."""
    budget = ReadBudget(limit=READ_LIMIT)
    read_first = ReadFirst(files=[str(BOK_ROOT / "README.md"), str(BOK_ROOT / "INDEX.md")])
    policy = Policy().add_graph({
        "plan_tasks": ["read_many"],
        "read_many": ["read_many", "read_images", final_tool],
        "read_images": ["read_many", "read_images", final_tool],
        final_tool: [],
    })
    policy = policy.add_chain(start=["plan_tasks"], end=[final_tool])
    return policy.add({
        "read_many": [Clause(check_multiple_files), Clause(read_first.check_first),
                      Clause(ReadOnce), Clause(budget.check_read)],
    })

In [ ]:
def local_image_as_b64(path: str) -> str:
    """Read a local image file and return a base64 data URI."""
    mime, _ = mimetypes.guess_type(path)
    if mime is None:
        mime = "image/png"
    with open(path, "rb") as f:
        data = f.read()
    uri = f"data:{mime};base64," + base64.b64encode(data).decode()
    print(f"  {os.path.basename(path):50s}  {mime}  {len(uri):,} chars")
    return uri

In [ ]:
def read_images(filepath: str):
    """Read images from a given file path"""
    messages = [
    {
        "role": "user",
        "content": [
            {"type": "text",      "text": "Describe this image giving as much detail as possible."},
            {"type": "image_url", "image_url": {"url": local_image_as_b64(filepath)}},
        ],
    }
    ]
    response = client.chat.completions.create(model=LLM_MODEL, messages=messages, max_tokens=20000)
    
    return response.choices[0].message.content.strip()

In [ ]:
# vector DB built by build_vector_index.ipynb (table: swebok_chunks (chunk | embedding))
SWEBOK_DB = REPO_ROOT / "swebok.duckdb"
SWEBOK_TABLE = "swebok_chunks"
SWEBOK_MODEL = "openrouter:qwen/qwen3-embedding-8b"  # must match EMBEDDING_MODEL in build_vector_index
SWEBOK_K = 10  # passages returned per search

# read the embedding dimension from the DB so it always matches what build_vector_index wrote
with duckdb.connect(str(SWEBOK_DB), read_only=True) as _con:
    SWEBOK_DIM = _con.execute(f"SELECT len(embedding) FROM {SWEBOK_TABLE} LIMIT 1").fetchone()[0]

swebok_embedder = AisuiteEmbeddingModel(SWEBOK_MODEL, batch_size=32, dimensions=SWEBOK_DIM)


def search_swebok(query: str) -> str:
    """Semantic search over the SWEBOK v4 (Guide to the Software Engineering Body of Knowledge).

    Use it to find the SWEBOK passages most relevant to a software engineering question.
    Write the query in English and be specific (e.g. "how to manage technical debt during maintenance").

    Args:
        query (str): natural-language question or topic to search for
    Returns:
        str: the top passages, ranked by relevance, each with its cosine distance (lower is closer)
    """
    q = np.asarray(swebok_embedder.encode([query], normalize_embeddings=True)[0], dtype=np.float32)
    with duckdb.connect(str(SWEBOK_DB), read_only=True) as con:
        con.execute("INSTALL vss; LOAD vss;")
        rows = con.execute(
            f"""
            SELECT array_cosine_distance(embedding, ?::FLOAT[{SWEBOK_DIM}]) AS distance, chunk
            FROM {SWEBOK_TABLE}
            ORDER BY distance
            LIMIT ?
            """,
            [q.tolist(), SWEBOK_K],
        ).fetchall()
    return "\n\n".join(
        f"[{i}] (distance={d:.4f})\n{chunk}" for i, (d, chunk) in enumerate(rows, 1)
    )

In [ ]:
EXPLORER_PROMPT = """
You are a research assistant that consults a Body of Knowledge (BoK) on behalf of a requester. You answer the requester's question using ONLY what you actually read in the BoK.

## The BoK

Name: {bok_name}
Location: {bok_path}

Layout:
- {bok_root}/README.md: overview of the whole BoK.
- {bok_root}/INDEX.md: list of the top-level topics, with links to each topic folder.
- Every topic folder has:
  - INDEX.md: links to the items one level below (sub-folders or content files).
  - README.md: summary of what that folder covers.
- Content files are the other .md files. They are the leaves of the tree and the only real evidence.
- Tables are in {bok_path}/tables and images are in {bok_path}/images. Content files point to them as
  [Table N](tables/<file>.md) and ![image](images/<file>.png). These links are relative to {bok_path}, not to the content file.
  These folders have no INDEX.md or README.md.
- Links inside INDEX.md are relative to the folder of that INDEX.md. Always turn them into full paths
  starting with {bok_root} before reading.

Top-level topics:
{topics}

## Tools

- plan_tasks: write your search plan. Always call it first.
- read_many: read several files in one call. Group the reads of each tree level into one call.
- read_images: describe one image file.
- write_file: write your full answer to {result_path}.

## Rules

1. Closed source. Every claim in your answer must come from a BoK passage you actually read in this run.
   Your own knowledge may only help you pick search terms and branches. It may not fill in content.
2. Read budget. Batches that contain only INDEX.md and README.md files are free.
   Batches that contain content files are limited to {read_limit}. Use the indexes to choose before you read content.
3. Do not map the whole BoK. Go down only the branches that are relevant to the question.
4. Titles, index entries and file names are for navigation only. They are not evidence.
5. If a content file you read points to a relevant table or image, read it. Do not guess its content from its name.
6. Never invent paths. Only use paths that appear in an INDEX.md you read.
7. If the BoK does not cover a point, say so plainly ("Not found in the BoK: <point>"). Do not fill the gap.
8. If two passages disagree, show both with their sources.

## Procedure

1. Plan: call plan_tasks with a short plan. List the topics to look for, not the answer.
2. Discover: in one read_many, read {bok_root}/README.md and {bok_root}/INDEX.md.
3. Navigate: for each relevant topic, read its INDEX.md and README.md (one read_many per level) and go down
   until you reach the content files that answer the question.
4. Read: read only the content files (and their tables or images) you need.
5. Check: for each statement you plan to write, confirm that a passage you read supports it in this context.
   Remove anything that fails the check.
6. Write: save the full answer with write_file to {result_path}. It must contain:
   - Answer: the guidance, organized by the points of the question.
   - Sources: for each statement, the file path (relative to {bok_path}) and the section, plus a short quote when it helps.
   - Gaps: the points the BoK does not cover.

If a tool fails or a path does not exist, report it. Do not pretend you read or wrote something.

## Final reply to the requester

After writing the file, reply ONLY in this format:

# Summary
2 to 4 sentences answering the question.

# Guidelines
- Guideline supported by the BoK (source: relative/path.md, section)
- ...

# Gaps
- Points the BoK does not cover, or "None".

Full answer: {result_path}
"""
topics_block = "\n".join(f"- {d.name}" for d in sorted(BOK_ROOT.iterdir()) if d.is_dir())
result_path = str(REPO_ROOT / "results" / "BOK_RESULT.md")
EXPLORER_PROMPT = EXPLORER_PROMPT.format(
    bok_name="SWEBOK v4 (Guide to the Software Engineering Body of Knowledge)",
    bok_path=BOK_PATH,
    bok_root=BOK_ROOT,
    topics=topics_block,
    read_limit=READ_LIMIT,
    result_path=result_path,
)

In [ ]:
explorer = create_agent(
    role=EXPLORER_PROMPT,
    policy=get_policy(),
    skills={
        "read_many": read_many,
        "read_images": read_images,
        "plan_tasks": plan_tasks,
        "write_file": write_file,
    },
    **AGENT_KWARGS,
)

In [ ]:
CODER_ROLE = ("You are a useful coding agent that creates codes based on SWEBOK. "
              "To check swebok, use the appropriate tool to give you the guidelines.")


def make_coder(**swebok_tool):
    return create_agent(role=CODER_ROLE, skills={"run_bash": run_bash, **swebok_tool}, **AGENT_KWARGS)


naive_coder = make_coder(search_swebok=search_swebok)  # vector search over swebok.duckdb
coder = make_coder(explorer=explorer)                  # explorer agent walking the BoK tree

## JEV over the SWEBOK

Three ways of using JEV (TypeSafe's decision model, through OpenRouter) on the same BoK tree:

| variant | who walks the tree | what JEV does |
|---|---|---|
| filter | the explorer (`read_many`) | judges every guideline against the refs it cites, dropping the unsupported ones |
| explorer | JEV, with a fixed question and judge | picks the branches at every `INDEX.md` and judges every leaf reached; an LLM writes the guidelines |
| tool | JEV, with the question and the judge the agent writes | the same walk, as one `jev_search` call of the coder |

In [ ]:
JEV_MODEL = "jev-latest"
JEV_BASE_URL = "https://openrouter.ai/api"

In [ ]:
def build_citations_json(recommendations_json, output_path=None):
    """{"recommendations": [{"recommendation", "refs": [paths]}]} → each recommendation with the
    content of the files it cites in 'refs'."""
    data = json.loads(recommendations_json)
    final_recs = {"recommendations": []}
    for rec in data["recommendations"]:
        final_recs["recommendations"].append({"recommendation": rec["recommendation"],
                                              "refs": read_many(rec["refs"])})
    if output_path is not None:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(final_recs, f, ensure_ascii=False, indent=2)
    return final_recs


def generate_final_recommendation(recommendations_json: str):
    """
    Generate the final guidelines.
    recommendations_json: the guidelines JSON with the schema:

    {
      "$schema": "http://json-schema.org/draft-07/schema#",
      "title": "RecommendationsResponse",
      "type": "object",
      "properties": {
        "recommendations": {
          "type": "array",
          "items": {
            "type": "object",
            "properties": {
              "recommendation": {"type": "string", "description": "The guideline"},
              "refs": {"type": "array", "description": "Paths of the BoK files that support it",
                       "items": {"type": "string"}}
            },
            "required": ["recommendation", "refs"],
            "additionalProperties": false
          }
        }
      },
      "required": ["recommendations"],
      "additionalProperties": false
    }
    """
    result_json = build_citations_json(recommendations_json)
    filtered = {"recommendations": []}
    with TypeSafeClient(api_key=os.environ["OPENROUTER_API_KEY"], base_url=JEV_BASE_URL) as ts:
        for rec in result_json["recommendations"]:
            result = ts.system_one(
                model=JEV_MODEL,
                state=rec,
                questions={"supported": Noul(
                    instructions="Can the guideline be stated from what the references say?",
                    criteria=NoulCriteria(
                        true="The references support the statement, directly or indirectly",
                        false="The statement has nothing to do with the references, it goes off topic",
                    ),
                )},
            )
            if result.nouls["supported"].noul > 0.5:
                filtered["recommendations"].append(rec)
    return filtered


JEV_FILTER_ROLE = EXPLORER_PROMPT + """
## Final step (replaces write_file)

Do not call write_file. Finish by calling generate_final_recommendation with every guideline and the
paths of the BoK files that support it; it drops the guidelines the references do not support.
Reply only with the guidelines it returns.
"""


def get_jev_filter_explorer():
    return create_agent(
        role=JEV_FILTER_ROLE,
        policy=get_policy(final_tool="generate_final_recommendation"),
        skills={
            "read_many": read_many,
            "read_images": read_images,
            "plan_tasks": plan_tasks,
            "generate_final_recommendation": generate_final_recommendation,
        },
        **AGENT_KWARGS,
    )

In [ ]:
class JevSearcher:
    """JEV's walk over the body of knowledge, as a tool: one call runs the whole pipeline."""

    LINK = re.compile(r'^- \[(.+?)\]\(<(.+?)>\)', re.M)
    TABLE = re.compile(r'\]\((tables/[^)]+\.md)\)')
    MAX_LABELS = 255  # JEV limit per Choice

    def __init__(self, bok_path, k=10, min_p=0.0, threshold=0.3, concurrency=16,
                 max_chars=60000, model=JEV_MODEL):
        self.bok_path = Path(bok_path)
        self.k, self.min_p, self.threshold = k, min_p, threshold
        self.concurrency, self.max_chars, self.model = concurrency, max_chars, model

    async def __call__(self, case: str, question: str, instructions: str,
                       true_criteria: str, false_criteria: str) -> str:
        """
        Search the body of knowledge for the documents that serve a task, without reading the tree yourself.

        One call runs the whole search. At every index of the body of knowledge, `question` picks the
        most promising branches; every document reached is judged by `instructions` with the true and
        false criteria; the approved documents come back, ranked by the judge's score, with their
        content. Be specific in both parts: a generic question prunes the wrong branches, and a generic
        criterion approves documents that change nothing in the task. For different aspects of the
        same task (design, testing, maintenance), run one search for each.

        Args:
            case: The full task, as JEV will see it at every decision.
            question: The question that picks the branches at every index, asked over the titles, e.g.:
                "Which of these branches most likely contain guidance on unit test design for this module?"
            instructions: The question the judge answers for every document, e.g.: "Does the document
                give practices or criteria for designing unit tests applicable to this task?"
            true_criteria: When the answer is yes: what the document must bring.
            false_criteria: When the answer is no: what disqualifies the document.
        """
        fields = {"case": case, "question": question, "instructions": instructions,
                  "true_criteria": true_criteria, "false_criteria": false_criteria}
        empty = [name for name, value in fields.items() if not value.strip()]
        if empty:
            return f"Error: {', '.join(empty)} empty. The five fields are required."
        judge = {"type": "noul", "instructions": instructions,
                 "criteria": {"true": true_criteria, "false": false_criteria}}
        async with AsyncTypeSafeClient(model=self.model, api_key=os.environ["OPENROUTER_API_KEY"],
                                       base_url=JEV_BASE_URL) as ts:
            judged = await self._walk(ts, case, question, judge)
        return self._report(judged)

    # ── the tree ──
    def _list_children(self, index_path: Path) -> dict:
        """INDEX.md → {title: target}; a target is a sub-folder INDEX.md or a leaf .md."""
        text = index_path.read_text(encoding="utf-8")
        return {t: index_path.parent / p for t, p in self.LINK.findall(text)
                if (index_path.parent / p).exists()}

    def _describe(self, target: Path):
        """A sub-folder is described by its INDEX (titles one level down); a leaf by its title alone."""
        return "\n".join(self._list_children(target)) if target.name == "INDEX.md" else None

    def _load_leaf(self, path: Path) -> str:
        """Leaf text with its tables inlined (table links are relative to the BoK folder)."""
        topic_dir = next(p for p in path.parents if p.name == "root").parent
        text = path.read_text(encoding="utf-8")
        tables = [topic_dir / t for t in dict.fromkeys(self.TABLE.findall(text))]
        return text + "".join(f"\n\n## {t.stem}\n{t.read_text(encoding='utf-8')}"
                              for t in tables if t.exists())

    def _start(self):
        """bok_path may be one BoK (has root/INDEX.md) or a folder of BoKs (one per sub-folder)."""
        if (self.bok_path / "root" / "INDEX.md").exists():
            start = self._list_children(self.bok_path / "root" / "INDEX.md")
        else:
            start = {d.name: d / "root" / "INDEX.md" for d in self.bok_path.iterdir()
                     if (d / "root" / "INDEX.md").exists()}
        if not start:
            raise FileNotFoundError(f"No root/INDEX.md under {self.bok_path}")
        return start

    # ── the walk ──
    async def _top_k(self, ts, sem, case, question, criteria):
        """Top-k labels of one Choice. Above MAX_LABELS: rank each slice, then the winners."""
        k, min_p = self.k, self.min_p
        if len(criteria) <= k and not min_p:
            return list(criteria)  # every one would pass anyway: skip the call
        if len(criteria) > self.MAX_LABELS:
            labels = list(criteria)
            chunks = [{l: criteria[l] for l in labels[i:i + self.MAX_LABELS]}
                      for i in range(0, len(labels), self.MAX_LABELS)]
            winners = await asyncio.gather(*(self._top_k(ts, sem, case, question, c) for c in chunks))
            return await self._top_k(ts, sem, case, question, {l: criteria[l] for w in winners for l in w})
        async with sem:
            res = await ts.system_one(
                state=case,
                questions={"pick": {"type": "choice", "instructions": question, "criteria": criteria}},
            )
        probs = res.choices["pick"].probabilities
        return [l for l in sorted(probs, key=probs.get, reverse=True)[:k] if probs[l] >= min_p]

    async def _walk(self, ts, case, question, judge):
        """Beam over the INDEX tree; each leaf is judged as soon as it is picked, while the rest of the
        tree is still being walked. Returns [(path, content, score)] for every leaf judged."""
        sem = asyncio.Semaphore(self.concurrency)
        judged = []

        async def judge_leaf(path):
            content = self._load_leaf(path)
            async with sem:
                res = await ts.system_one(state={"task": case, "document": content},
                                          questions={"useful": judge})
            return path, content, res.nouls["useful"].noul

        async def descend(children):
            criteria = {t: self._describe(p) for t, p in children.items()}
            picked = [children[t] for t in await self._top_k(ts, sem, case, question, criteria)]
            judged.extend(asyncio.create_task(judge_leaf(p)) for p in picked if p.name != "INDEX.md")
            await asyncio.gather(*(descend(self._list_children(p)) for p in picked if p.name == "INDEX.md"))

        await descend(self._start())
        return await asyncio.gather(*judged)

    # ── the answer ──
    def _report(self, judged):
        """The approved ones by score, with content, up to max_chars; and what was left out, so the
        agent knows whether to search again with another question or criterion."""
        judged = sorted(judged, key=lambda d: d[2], reverse=True)
        passed = [d for d in judged if d[2] >= self.threshold]
        head = f"{len(judged)} documents judged, {len(passed)} approved (score >= {self.threshold})."
        if not passed:
            near = "\n".join(f"- [{s:.2f}] {p}" for p, _, s in judged[:5])
            return (f"{head} No document passed. The closest ones:\n{near}\n"
                    "Search again with a question or criterion closer to what these documents cover.")
        blocks, used, cut = [], 0, 0
        for path, content, score in passed:
            block = f"## [{score:.2f}] {path}\n{content}"
            if used + len(block) > self.max_chars and blocks:
                cut += 1
                continue
            blocks.append(block)
            used += len(block)
        tail = f" {cut} approved documents left out by the {self.max_chars}-character cap." if cut else ""
        return head + tail + "\n\n" + "\n\n".join(blocks)

In [ ]:
CHOOSE = "Which of these branches of the body of knowledge most likely contain guidance for this task?"
USEFUL = {
    "instructions": "Does the document contain information useful to carry out this task?",
    "true_criteria": "The document brings practices, criteria or data directly applicable to the task",
    "false_criteria": "The document covers another subject or changes nothing in how the task is done",
}
RECOMMEND_PROMPT = """
You are a exploration tool. Based ONLY on the references given, write the guidelines for the task
in this format:

# Summary

Up to 500 characters.

# Guidelines

- guideline (source: relative/path.md)
- ...

Use only the text you receive. Never add guidelines from outside it.
"""


async def jev_explore(task: str):
    """JEV walks the tree with the fixed question and judge; the approved documents go to the LLM."""
    references = await JevSearcher(BOK_PATH)(task, CHOOSE, **USEFUL)
    messages = [
        {"role": "system", "content": RECOMMEND_PROMPT},
        {"role": "user", "content": f"Task:\n{task}\n\nReferences:\n{references}"},
    ]
    response = client.chat.completions.create(model=LLM_MODEL, messages=messages, max_tokens=10000)
    return response.choices[0].message.content.strip()

In [ ]:
class CallBudget:
    """At most `limit` calls of the tool it guards."""
    def __init__(self, limit=1):
        self.limit = limit
        self.count = 0

    def __call__(self, **kwargs):
        self.count += 1
        if self.count > self.limit:
            raise PermissionError("Call limit reached! Now you must finish your task at once!")

def get_jev_explorer_free(limit=1):
    JEV_CODER_ROLE = """You are a searcher agent that writes practices following the SWEBOK. To do so, you use the 
    `jev_seach` tool. To proper use it, ask it broad questions and define mutually exclusive and non-ambiguous acceptance """
    return create_agent(
        role=JEV_CODER_ROLE,
        policy=Policy().add({"jev_search": [Clause(CallBudget(limit=limit))]}),
        skills={"jev_explore": JevSearcher(BOK_PATH)},
        **AGENT_KWARGS,
    )

def get_jev_explorer(limit=1):
    return create_agent(
        role=CODER_ROLE,
        policy=Policy().add({"jev_explore": [Clause(CallBudget(limit=READ_LIMIT))]}),
        skills={'run_bash': run_bash, "jev_explore": jev_explore},
        **AGENT_KWARGS,
    )

In [ ]:
coder_jev_searcher = get_jev_explorer()
coder_jev_searcher_free = get_jev_explorer()
coder_jev_filter =  get_jev_filter_explorer()

In [ ]:
AgentDemo(coder, title="Explorer").launch()

In [ ]:
AgentDemo(naive_coder, title="Agentic RAG").launch()

In [ ]:
AgentDemo(coder_jev_searcher, title="GROVE").launch()

In [18]:
await jev_explore('regression tests')

"# Summary\n\n[Guide wording omitted] system under test ([Guide wording omitted]. It can be conducted at any test level, is fundamental to Agile/DevOps/TDD practices, and may involve test case selection, minimization, and prioritization strategies.\n\n# Guidelines\n\n- Perform regression testing [Guide wording omitted] (source: root/Software Testing Levels, Objectives, and Approaches/Alpha, Beta, and Regression Testing Procedures.md)\n- Ensure [Guide wording omitted] after each change (source: root/Software Testing Levels, Objectives, and Approaches/Alpha, Beta, and Regression Testing Procedures.md)\n- Balance the assurance provided by regression testing against the resources required, as running all tests after every change can be time-consuming (source: root/Software Testing Levels, Objectives, and Approaches/Alpha, Beta, and Regression Testing Procedures.md)\n- Conduct regression testing at any applicable test level, including unit, integration, system, and acceptance (source: root/